In [4]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('D:\Dev\Point-Cloud-Registration\source')

import pickle
import os
import gc
from copy import deepcopy
import pandas as pd
import numpy as np
from glob import glob
from pathlib import Path
import matplotlib.pyplot as plt
from multiprocessing import Pool
import open3d as o3d
import time

from cylinder_alignment.bundle_station import BundleStation
from cylinder_extraction.station import Station
from cylinder_alignment.station_alignment import StationAlignment
from cylinder_alignment.utils import extract_target_cylinder
from utils.write_output import write_array2txt
from utils.math_methods import calc_overlap_curve

import plotly.express as px

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


5 mins
5


In [6]:
scan_field_names = [
    ('JB-448 26ft EXT Faro',8),
    ('JB-448 26ft EXT Faro/tilted stations',8),
    ('JB-567 27ft EXT Leica/tilted stations',8),
    ('JB-531 40ft EXT',8),
    ('JB-453 60ft EXT Faro',8),
    ('JB-550 90ft EXT Trimble',2),
    ('JB-505 100ft EXT Trimble',2),
    ('JB-595 117ft EXT - FARO',8),
    ('JB-566 120ft EXT',8)
]
execuated_time = {}
for field, pack_num in scan_field_names[8:9]:    
    time_begin = time.time()
    try:
        print('===========================')
        print(field)
        data_dir = Path(f'D:/REGISTRATION DATA/External/{field}')
        # READ EXTRACTED CYLINDERS DATA
        bundle = BundleStation(data_dir=data_dir)
        summary_df = bundle.read_extracted_cylinder_data(file_types=['e57', 'pcd'])
        # ############################
        # there are some specilities with some tanks that need to be set manually
        if field == 'JB-531 40ft EXT':
            # roof stations 001, 002, 003
            valid_stations = []
            for s in bundle.stations:
                if '001' in s.name or '002' in s.name or '003' in s.name:
                    continue
                valid_stations.append(s)
            bundle.stations = valid_stations

        if field == 'JB-567 27ft EXT Leica/tilted stations':
            # roof stations 001, 002, 003
            valid_stations = []
            for s in bundle.stations:
                if '001' in s.name or '002' in s.name or '003' in s.name:
                    continue
                valid_stations.append(s)
            bundle.stations = valid_stations

        if field == 'JB-566 120ft EXT':
            # not-understandable station data: setup 030
            valid_stations = []
            for s in bundle.stations:
                if '030' in s.name:
                    continue
                valid_stations.append(s)
            bundle.stations = valid_stations
        # px.scatter(
        #     summary_df[summary_df.radius < 50], y='radius', x='station', 
        #     color='station', hover_data=summary_df.columns
        # )
        # PAIRWISE STATION ALIGNMENT
        bundle.align_pairwise_station()
        time_pairwise = time.time()
        # Save
        bundle.remove_points()
        with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'wb') as f:
            pickle.dump(bundle, f)
        # load
        with open(os.path.join(data_dir, 'rough_alignment.pkl'), 'rb') as f:
            bundle = pickle.load(f)
        # uncertain stations
        uncertain_stations = bundle.find_uncertain_stations()
        if len(uncertain_stations) > 0:
            print('uncertain stations')
            print(uncertain_stations)
            raise ValueError('Uncertain stations exist')
        # bundle.handle_uncertain(
        #     neighbor_info={
        #         '009': '008',
        #         '016': '015',
        #         '018': '017'
        #     }
        # )
        # bring all stations to a global coordinate
        bundle.align_all_stations_by_axis()
        # CLUSTER TANKS
        bundle.cluster_cylinder()
        time_cluster = time.time()
        # save
        bundle.remove_points()
        with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'wb') as f:
            pickle.dump(bundle, f)
        with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'rb') as f:
            bundle = pickle.load(f)
        # VISUAL INSPECTION
        # fig = bundle.draw_cross_section()
        # fig.show()
        # TARGET TANK
        bundle.identify_target_tank()
        print('target tank', bundle.target_tank_id)
        # bundle.set_target_tank(tank_id=3)
        # Put all fragments of the target tank in one place. Include points at their foot
        bundle.write_target_tank(
            write_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground',
            load_data=True, extend_ground=True, transform=False)
        # update location of all the segments of the target tank: load original ground-extended data
        for c in bundle.target_tank:
            c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
            print(c.ptc_filepath)

        # alignment transformation matrices
        bundle.calc_align_matrix(f'{bundle.target_tank_id}')
        # save
        bundle.remove_points()
        with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'wb') as f:
            pickle.dump(bundle, f)
        with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
            bundle = pickle.load(f)

        # write the transformed segments
        bundle.write_target_tank(
            write_dir=bundle.data_dir / 'target_tank' / 'global' / 'ground', 
            load_data=True, transform=True, extend_ground=False, num_process=5)

        # THIN SLICE
        bundle.cut_horizontal(
            mode='station_cylinders', z_cut=3, thickness=0.01, 
            write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
        bundle.cut_horizontal(
            mode='target_tank', z_cut=3, thickness=0.01,
            write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
    except Exception as exc:
        print(f'===== FAILED HANDLING SCAN {field} =====')
        print(exc)
    time_end = time.time()
    lasted = time_end - time_begin
    print(f'processing took {lasted} seconds')
    execuated_time[field] = lasted
    del bundle
    gc.collect()
        

JB-566 120ft EXT
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 014\Plains Tank 401- Setup 014.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 015\Plains Tank 401- Setup 015.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 016\Plains Tank 401- Setup 016.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 017\Plains Tank 401- Setup 017.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 018\Plains Tank 401- Setup 018.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 019\Plains Tank 401- Setup 019.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 020\Plains Tank 401- Setup 020.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 021\Plains Tank 401- Setup 021.pkl
D:\REGISTRATION DATA\External\JB-566 120ft EXT\Plains Tank 401- Setup 022\Plains Tank 401- Setup 022.pkl
D:\REGISTRATION DATA\External\JB-566 1

In [19]:
execuated_time

{'JB-505 100ft EXT Trimble': 2652.029712677002}

In [2]:
data_dir = Path(f'D:/REGISTRATION DATA/External/JB-505 100ft EXT Trimble')

In [3]:
with open(os.path.join(data_dir, 'globalization_center_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

In [22]:
a = np.random.rand(3)
a = np.sort(a)
a

array([0.26259395, 0.32570422, 0.80599012])

In [23]:
b = np.r_[0, a, 2*np.pi]

In [24]:
b

array([0.        , 0.26259395, 0.32570422, 0.80599012, 6.28318531])

In [27]:
np.sum(np.diff(b)) / (2*np.pi)

1.0

In [4]:
bundle.identify_target_tank()

span [0.12387674996013776, 0.39200815263957156, 0.502873791841946, 0.5919053703393555, 0.5262141485156824, 0.3577527339922416, 0.4301320760213696, 0.24500650995045892, 0.4593352369873045, 0.5269882434126241, 0.2492767008364698, 0.5284045009185527, 0.5554518819137297, 0.2598230979333286, 0.10016649098216446, 1.0, 0.22784227191411477, 0.6242515181131789, 0.09525991445317783]


15

In [5]:
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground',
    load_data=True, extend_ground=True, transform=False)
# update location of all the segments of the target tank: load original ground-extended data
for c in bundle.target_tank:
    c.update_filepath(new_dir=bundle.data_dir / 'target_tank' / 'local' / 'ground', num_keep_level=1, file_name=f'{Path(c.ptc_filepath).stem}.pcd')
    print(c.ptc_filepath)

# alignment transformation matrices
bundle.calc_align_matrix(f'{bundle.target_tank_id}')
# save
bundle.remove_points()
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'wb') as f:
    pickle.dump(bundle, f)
with open(os.path.join(data_dir, 'globalization_axis_phase_1.pkl'), 'rb') as f:
    bundle = pickle.load(f)

# write the transformed segments
bundle.write_target_tank(
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'ground', 
    load_data=True, transform=True, extend_ground=False, num_process=5)

# THIN SLICE
bundle.cut_horizontal(
    mode='station_cylinders', z_cut=3, thickness=0.01, 
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')
bundle.cut_horizontal(
    mode='target_tank', z_cut=3, thickness=0.01,
    write_dir=bundle.data_dir / 'target_tank' / 'global' / 'thin_slice')

MemoryError: Unable to allocate 3.10 GiB for an array with shape (3, 138744384) and data type float64